In [ ]:
# Import the libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Read the dataset
original_DF = pd.read_csv("netflix_titles.csv")


In [ ]:
# Display the first few records
original_DF.head()


In [ ]:
# Display number of rows and columns
original_DF.shape


In [ ]:
# Display the data types of each column
original_DF.dtypes


In [ ]:
# Display the summary statistics of the dataset
original_DF.describe(include="all")


In [ ]:
# Checking duplicate rows
exact_duplicates = original_DF.duplicated().sum()
print(f"Number of exact duplicate rows: {exact_duplicates}")

duplicate_show_ids = original_DF["show_id"].duplicated().sum()
print(f"Number of duplicate show IDs: {duplicate_show_ids}")


In [ ]:
# Create a working copy and standardize whitespace
cleansed_DF = original_DF.copy()

for column in cleansed_DF.select_dtypes(include="object").columns:
    cleansed_DF[column] = cleansed_DF[column].str.strip()


In [ ]:
# Identify duration values incorrectly stored in the rating column
misaligned_mask = cleansed_DF["rating"].str.contains(r"^\d+\s+min$", na=False)

cleansed_DF.loc[misaligned_mask, ["show_id", "title", "rating", "duration"]]


In [ ]:
# Correct the misaligned duration entries
cleansed_DF.loc[misaligned_mask, "duration"] = cleansed_DF.loc[misaligned_mask, "rating"]
cleansed_DF.loc[misaligned_mask, "rating"] = np.nan

print(f"Number of corrected records: {misaligned_mask.sum()}")


In [ ]:
# Removing exact duplicate rows
cleansed_DF = cleansed_DF.drop_duplicates().copy()
print(f"Number of rows after removing duplicates: {cleansed_DF.shape[0]}")


In [ ]:
# Checking missing values before filling
cleansed_DF.isnull().sum()


In [ ]:
# Convert date_added to datetime
cleansed_DF["date_added"] = pd.to_datetime(cleansed_DF["date_added"], errors="coerce")


In [ ]:
# Fill missing categorical and text values
categorical_fill_cols = ["director", "cast", "country", "rating", "duration"]

for column in categorical_fill_cols:
    cleansed_DF[column] = cleansed_DF[column].fillna("Unknown")

print("Missing values after filling:")
cleansed_DF.isnull().sum()


In [ ]:
# Creating new columns for analysis
cleansed_DF["date_added_year"] = cleansed_DF["date_added"].dt.year
cleansed_DF["date_added_month"] = cleansed_DF["date_added"].dt.month_name()

cleansed_DF["duration_value"] = pd.to_numeric(
    cleansed_DF["duration"].str.extract(r"(\d+)")[0], errors="coerce"
)

cleansed_DF["duration_unit"] = np.select(
    [
        cleansed_DF["duration"].str.contains("min", case=False, na=False),
        cleansed_DF["duration"].str.contains("season", case=False, na=False)
    ],
    ["Minutes", "Seasons"],
    default="Unknown"
)

cleansed_DF["primary_genre"] = cleansed_DF["listed_in"].str.split(",").str[0].str.strip()

cleansed_DF["primary_country"] = (
    cleansed_DF["country"]
    .replace("Unknown", np.nan)
    .str.split(",").str[0].str.strip()
    .fillna("Unknown")
)

cleansed_DF.head()


In [ ]:
# Checking inconsistent entries and category frequencies
categorical_cols = ["type", "rating", "duration_unit", "primary_genre"]

for column in categorical_cols:
    print(f"\\n{column}")
    print(cleansed_DF[column].value_counts(dropna=False).head(20))


In [ ]:
# Checking numerical ranges
numerical_cols = ["release_year", "date_added_year", "duration_value"]

for column in numerical_cols:
    print(f"\\n{column}")
    print(cleansed_DF[column].describe())


In [ ]:
# Verify missing values and duplicate rows after cleaning
print("Missing values:")
print(cleansed_DF.isnull().sum())

print("\\nExact duplicate rows:", cleansed_DF.duplicated().sum())


In [ ]:
# Check leading and trailing whitespace in the original text fields
for column in original_DF.select_dtypes(include="object").columns:
    whitespace_count = (
        original_DF[column].dropna() != original_DF[column].dropna().str.strip()
    ).sum()
    if whitespace_count > 0:
        print(f"{column}: {whitespace_count} values contain extra whitespace")


In [ ]:
# Check for unexpected values in key categorical columns
valid_types = {"Movie", "TV Show"}
valid_ratings = {
    "G", "NC-17", "NR", "PG", "PG-13", "R",
    "TV-14", "TV-G", "TV-MA", "TV-PG", "TV-Y", "TV-Y7", "TV-Y7-FV", "UR"
}

invalid_types = cleansed_DF[~cleansed_DF["type"].isin(valid_types)]
invalid_ratings = cleansed_DF[
    (cleansed_DF["rating"] != "Unknown") &
    (~cleansed_DF["rating"].isin(valid_ratings))
]

print("Unexpected type values:", len(invalid_types))
print("Unexpected rating values:", len(invalid_ratings))


In [ ]:
# Check duration format
duration_pattern = r"^\\d+\\s+(min|season|seasons)$"
invalid_duration = cleansed_DF[
    (cleansed_DF["duration"] != "Unknown") &
    (~cleansed_DF["duration"].str.match(duration_pattern, case=False, na=False))
]

print("Malformed duration values:", len(invalid_duration))
invalid_duration[["show_id", "title", "type", "duration"]].head()


In [ ]:
# Check whether duration units match the content type
movie_with_seasons = cleansed_DF[
    (cleansed_DF["type"] == "Movie") &
    (cleansed_DF["duration_unit"] == "Seasons")
]
tv_show_with_minutes = cleansed_DF[
    (cleansed_DF["type"] == "TV Show") &
    (cleansed_DF["duration_unit"] == "Minutes")
]

print("Movies with season-based duration:", len(movie_with_seasons))
print("TV Shows with minute-based duration:", len(tv_show_with_minutes))


In [ ]:
# Check unique show identifiers
print("Duplicate show IDs:", cleansed_DF["show_id"].duplicated().sum())


In [ ]:
# Check date-added values that appear earlier than the release year
# This is a review flag, not automatically an error: release_year may represent
# the original release or a show's final release year.
date_year_check = cleansed_DF[
    cleansed_DF["date_added"].notna() &
    (cleansed_DF["date_added"].dt.year < cleansed_DF["release_year"])
][["show_id", "title", "release_year", "date_added"]]

print("Date-added year earlier than release year:", len(date_year_check))
date_year_check.head(20)


## Univariate Analysis

Univariate analysis explores one variable at a time to understand the distribution of Netflix content.


In [ ]:
# Univariate Analysis for categorical variables
# Bar charts
for column in ["type", "rating", "duration_unit"]:
    plt.figure(figsize=(10, 5))
    cleansed_DF[column].value_counts().head(15).plot(kind="bar")
    plt.xlabel(column.replace("_", " ").title())
    plt.ylabel("Number of Titles")
    plt.title(f"Distribution of {column.replace('_', ' ').title()}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Top 15 genres
plt.figure(figsize=(12, 6))
cleansed_DF["primary_genre"].value_counts().head(15).sort_values().plot(kind="barh")
plt.xlabel("Number of Titles")
plt.ylabel("Primary Genre")
plt.title("Top 15 Primary Genres on Netflix")
plt.tight_layout()
plt.show()


In [ ]:
# Univariate analysis for numerical variables
cleansed_DF[["release_year", "duration_value"]].hist(figsize=(12, 5), bins=20)
plt.suptitle("Histograms of Numerical Variables")
plt.tight_layout()
plt.show()


## Bivariate Analysis

### Numerical vs Numerical


In [ ]:
# Release Year vs Duration Value
# Scatter plot
movie_DF = cleansed_DF[
    (cleansed_DF["type"] == "Movie") &
    (cleansed_DF["duration_unit"] == "Minutes")
]

plt.figure(figsize=(12, 6))
plt.scatter(movie_DF["release_year"], movie_DF["duration_value"], alpha=0.5)
plt.xlabel("Release Year")
plt.ylabel("Duration (Minutes)")
plt.title("Release Year vs Movie Duration")
plt.tight_layout()
plt.show()


### Numerical vs Categorical


In [ ]:
# Release Year vs Content Type
plt.figure(figsize=(10, 6))
cleansed_DF.boxplot(column="release_year", by="type")
plt.xlabel("Content Type")
plt.ylabel("Release Year")
plt.title("Release Year vs Content Type")
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
# Movie Duration vs Rating
plt.figure(figsize=(12, 6))
movie_rating_DF = movie_DF[movie_DF["rating"] != "Unknown"]
movie_rating_DF.boxplot(column="duration_value", by="rating", rot=45)
plt.xlabel("Rating")
plt.ylabel("Duration (Minutes)")
plt.title("Movie Duration vs Rating")
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
# Release Year vs Primary Genre (top 10 genres)
top_genres = cleansed_DF["primary_genre"].value_counts().head(10).index
genre_DF = cleansed_DF[cleansed_DF["primary_genre"].isin(top_genres)]

plt.figure(figsize=(14, 6))
genre_DF.boxplot(column="release_year", by="primary_genre", rot=45)
plt.xlabel("Primary Genre")
plt.ylabel("Release Year")
plt.title("Release Year vs Top 10 Primary Genres")
plt.suptitle("")
plt.tight_layout()
plt.show()


### Categorical vs Categorical


In [ ]:
# Content Type vs Rating
plt.figure(figsize=(12, 7))
cross_tab = pd.crosstab(cleansed_DF["type"], cleansed_DF["rating"])
sns.heatmap(cross_tab, annot=True, cmap="Blues", fmt="d")
plt.xlabel("Rating")
plt.ylabel("Content Type")
plt.title("Content Type vs Rating")
plt.tight_layout()
plt.show()


In [ ]:
# Content Type vs Primary Genre (top 12 genres)
top_genres = cleansed_DF["primary_genre"].value_counts().head(12).index
genre_cross_tab = pd.crosstab(
    cleansed_DF[cleansed_DF["primary_genre"].isin(top_genres)]["primary_genre"],
    cleansed_DF[cleansed_DF["primary_genre"].isin(top_genres)]["type"]
)

plt.figure(figsize=(10, 7))
sns.heatmap(genre_cross_tab, annot=True, cmap="Greens", fmt="d")
plt.xlabel("Content Type")
plt.ylabel("Primary Genre")
plt.title("Primary Genre vs Content Type")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix
correlation_cols = ["release_year", "date_added_year", "duration_value"]
corr_matrix = cleansed_DF[correlation_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Outlier analysis using the IQR method
outlier_cols = ["release_year", "duration_value"]

for column in outlier_cols:
    values = cleansed_DF[column].dropna()
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = values[(values < lower_bound) | (values > upper_bound)]

    print(f"\\n{column}")
    print(f"Lower bound: {lower_bound}")
    print(f"Upper bound: {upper_bound}")
    print(f"Potential outliers: {len(outliers)}")


In [ ]:
# Box plots for numerical variables
for column in outlier_cols:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=cleansed_DF[column])
    plt.title(f"Box Plot of {column.replace('_', ' ').title()}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Export the cleaned dataset
cleansed_DF.to_csv("netflix_titles_cleaned.csv", index=False)
print("Cleaned dataset exported successfully.")
